# LPM24 GFlowNet v2.4.1 Two-GPU Ablation On Kaggle

Parameterized Kaggle notebook for the LPM24 multi-molecule GFlowNet v2.4.1 ablation. It clones `gflownet_v2.4.1` into `/kaggle/working/Thesis`, prepares the upstream SFT checkpoint and LPM24 grouped splits, trains with the selected two-GPU parallel ablation knobs, evaluates validation generations, and exports versioned Kaggle artifacts under `/kaggle/working/thesis_artifacts`.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/mruniverse8/Thesis.git"
REPO_BRANCH = "gflownet_v2.4.1"
REPO_DIR = Path("/kaggle/working/Thesis")


def run_stream(command, *, cwd=None):
    command = [str(part) for part in command]
    print("Running:", " ".join(command), flush=True)
    process = subprocess.Popen(
        command,
        cwd=str(cwd) if cwd is not None else None,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    if process.stdout is None:
        raise RuntimeError("Failed to capture command output.")

    try:
        for line in process.stdout:
            print(line, end="", flush=True)
    finally:
        process.stdout.close()

    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)


def clone_or_fetch_checkout(repo_url: str, repo_branch: str, repo_dir: Path) -> Path:
    repo_dir = Path(repo_dir)
    if (repo_dir / ".git").exists():
        print(f"Reusing {repo_dir}")
    elif repo_dir.exists():
        raise RuntimeError(f"Existing non-git directory at {repo_dir}; delete it and rerun the notebook.")
    else:
        run_stream(["git", "clone", "--depth", "1", repo_url, str(repo_dir)])

    run_stream(["git", "-C", str(repo_dir), "fetch", "--depth", "1", repo_url, repo_branch])
    run_stream(["git", "-C", str(repo_dir), "checkout", "-B", repo_branch, "FETCH_HEAD"])
    return repo_dir


%cd /kaggle/working
clone_or_fetch_checkout(REPO_URL, REPO_BRANCH, REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from thesis_kaggle_support import (
    create_zip_archive,
    ensure_paths_exist,
    export_stage_artifacts,
    json_dumps,
    read_json,
    report_runtime,
)

print(json_dumps({"repo_dir": str(REPO_DIR), "repo_branch": REPO_BRANCH}))

In [ ]:
from datetime import datetime

GFLOWNET_VERSION = "gflownet_v2.4.1"
STAGE_NAME = "train_gflownet_lpm24_v241_ablation"
ABLATION_NAME = "replay_tb_mixture_db_sample"
DATASET_MODE = "never"
CONFIG_OVERRIDE = Path("configs/multi_molecule_gflownet_lpm24.yaml")
CONFIG_STEM = CONFIG_OVERRIDE.stem
GFLOWNET_OBJECTIVE = "db"

# Top-level ablation knobs.
# Replay and target guidance are mutually exclusive in the trainer.
# For replay ablations, keep TARGET_GUIDANCE_ENABLED = False.
TARGET_GUIDANCE_ENABLED = False
TARGET_GUIDANCE_ON_POLICY_FRACTION = 0.25
TARGET_GUIDANCE_PREFIX_FRACTION = 0.50
TARGET_GUIDANCE_TEACHER_FRACTION = 0.25
TARGET_GUIDANCE_SHUFFLE_TARGET_SELFIES_LIST = True
REPLAY_ENABLED = True
REPLAY_BUFFER_TYPE = "experimental_tb_mixture"
REPLAY_FRACTION = 0.75
REPLAY_WITH_REPLACEMENT = True
REPLAY_CAPACITY = 157772
REPLAY_RECENT_FRACTION = 0.30
REPLAY_REWARD_FRACTION = 0.30
REPLAY_UNIFORM_FRACTION = 0.20
REPLAY_TB_RESIDUAL_FRACTION = 0.20
REPLAY_REWARD_TEMPERATURE = 1.0
REPLAY_TB_RESIDUAL_TEMPERATURE = 1.0
REPLAY_RECENT_WINDOW_SIZE = 64
REPLAY_MAX_INVALID_FRACTION = 0.20
REPLAY_MAX_DUPLICATE_FRACTION = 0.10
ROLLOUT_DECODING_STRATEGY = "sample"
ROLLOUT_NUM_BEAMS = 2
ROLLOUT_LENGTH_PENALTY = 1.0
ROLLOUT_EARLY_STOPPING = True
REWARD_VARIANT = "reward_var2"
ENABLE_INVALID_SIMILARITY_NGRAM_FALLBACK = False
PARALLEL_TRAINING_ENABLED = True
PARALLEL_TRAINING_MODE = "replicated_scoring"
PARALLEL_TRAINING_DEVICES = ["cuda:0", "cuda:1"]
PARALLEL_TRAINING_STRICT = True
GFLOWNET_CHECKPOINT_DOWNLOAD_SOURCE = "1jCIVYbzgTw7xQAWvv6SfwM8Y1vL47PDg"

GFLOWNET_ITERATIONS = 35000
GFLOWNET_BATCH_SIZE = 6
MAX_OPTIMIZATION_TRAJECTORIES_PER_ITER = 64
GFLOWNET_SCORING_MICROBATCH_SIZE = 16
GFLOWNET_LEARNING_RATE = 1e-6
GFLOWNET_WARMUP_RATIO = 0.03
GFLOWNET_SAVE_EVERY_ITERATIONS = 500
GFLOWNET_INVALID_TERMINAL_REWARD = 4.0e-2
REWARD_PENALTY_INVALID = 0.5
VALIDATION_FRACTION = 0.05
SPLIT_SEED = 42
MAX_TARGET_SYMBOLS = 1024
MAX_STAGE_SYMBOLS = 192
FORCE_VALID_MASKING = True
SELFIES_DICT_PATH = "molecules/dict/selfies_dict.txt"
ROLLOUT_MAX_STAGE_NEW_TOKENS = 192
ROLLOUT_MAX_MOLECULES_PER_SEQUENCE = 8
ROLLOUT_MAX_SEQUENCE_LENGTH = 4096
ROLLOUT_TEMPERATURE = 0.08
ROLLOUT_TOP_P = 0.25
ROLLOUT_APPEND_PROBABILITY = 0.30
ROLLOUT_INVALID_APPEND_PROBABILITY = 0.0
TERMINATE_ON_INVALID_STAGE = True
REWARD_DIVERSITY_BETA = 0.6
REWARD_DIVERSITY_WEIGHT = 0.4
REWARD_MATCH_WEIGHT = 1.0
REWARD_PLUS_VALID = 0.8

BASE_TRAIN_CONFIG = (REPO_DIR / CONFIG_OVERRIDE).resolve()
RUN_STAMP = datetime.now().strftime("%y%m%d_%H%M%S")
RUN_NAME = f"{CONFIG_STEM}_v241_{ABLATION_NAME}_{RUN_STAMP}"
OUTPUT_DIR = REPO_DIR / "outputs" / "kaggle" / STAGE_NAME / RUN_NAME
TEMP_CONFIG_PATH = REPO_DIR / "kaggle" / "generated_configs" / f"{RUN_NAME}.yaml"
ARTIFACT_OUTPUT_ROOT = Path("/kaggle/working/thesis_artifacts")
STAGE_ARTIFACT_ZIP = ARTIFACT_OUTPUT_ROOT / f"{STAGE_NAME}.zip"
RUN_SUMMARY_PATH = OUTPUT_DIR / "run_summary.json"
CHECKPOINTS_DIR = OUTPUT_DIR / "checkpoints"
BEST_CHECKPOINT_DIR = CHECKPOINTS_DIR / "best"
BEST_CHECKPOINT_ZIP = CHECKPOINTS_DIR / "best.zip"
ITERATION_DIAGNOSTICS_PATH = OUTPUT_DIR / "diagnostics" / "iteration_diagnostics.jsonl"
TRAJECTORY_PREVIEWS_PATH = OUTPUT_DIR / "diagnostics" / "trajectory_previews.jsonl"
GFLOWNET_REPORT_METRICS_PATH = OUTPUT_DIR / "diagnostics" / "gflownet_report_metrics.jsonl"
EVAL_OUTPUT_PATH = OUTPUT_DIR / "diagnostics" / "validation_evaluation_metrics.json"
UPSTREAM_CHECKPOINT = REPO_DIR / "outputs" / "multi_molecule_sft_lpm24" / "checkpoints" / "best"
LPM24_DATASET_DIR = REPO_DIR / "data" / "lpm24"
GROUPED_SPLITS_DIR = LPM24_DATASET_DIR / "grouped_splits"
PROCESSED_DATASET_CHECKS = {
    "train_multimol": LPM24_DATASET_DIR / "processed" / "train_multimol.jsonl",
    "test_multimol": LPM24_DATASET_DIR / "processed" / "test_multimol.jsonl",
    "test_eval_first_1000": LPM24_DATASET_DIR / "processed" / "test_eval_first_1000_multimol.jsonl",
}
GROUPED_SPLIT_CHECKS = {
    "train_multimol": GROUPED_SPLITS_DIR / "train_multimol.jsonl",
    "validation_multimol": GROUPED_SPLITS_DIR / "validation_multimol.jsonl",
    "test_multimol": GROUPED_SPLITS_DIR / "test_multimol.jsonl",
}

print(json_dumps({
    "gflownet_version": GFLOWNET_VERSION,
    "repo_branch": REPO_BRANCH,
    "stage_name": STAGE_NAME,
    "ablation_name": ABLATION_NAME,
    "base_train_config": str(BASE_TRAIN_CONFIG),
    "run_name": RUN_NAME,
    "output_dir": str(OUTPUT_DIR),
    "temp_config_path": str(TEMP_CONFIG_PATH),
    "artifact_zip": str(STAGE_ARTIFACT_ZIP),
    "gflownet_objective": GFLOWNET_OBJECTIVE,
    "gflownet_iterations": GFLOWNET_ITERATIONS,
    "gflownet_batch_size": GFLOWNET_BATCH_SIZE,
    "rollout_decoding_strategy": ROLLOUT_DECODING_STRATEGY,
    "rollout_num_beams": ROLLOUT_NUM_BEAMS,
    "target_guidance_enabled": TARGET_GUIDANCE_ENABLED,
    "replay_enabled": REPLAY_ENABLED,
    "replay_buffer_type": REPLAY_BUFFER_TYPE,
    "replay_fraction": REPLAY_FRACTION,
    "replay_source_fractions": {
        "recent": REPLAY_RECENT_FRACTION,
        "reward": REPLAY_REWARD_FRACTION,
        "uniform": REPLAY_UNIFORM_FRACTION,
        "tb_residual": REPLAY_TB_RESIDUAL_FRACTION,
    },
    "reward_variant": REWARD_VARIANT,
    "parallel_training_enabled": PARALLEL_TRAINING_ENABLED,
    "parallel_training_devices": PARALLEL_TRAINING_DEVICES,
    "checkpoint_source_configured": bool(GFLOWNET_CHECKPOINT_DOWNLOAD_SOURCE.strip()),
}))

In [ ]:
import os

WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "")
try:
    import wandb
except Exception as exc:
    wandb = None
    print({"wandb_import": f"unavailable:{exc.__class__.__name__}"})

if WANDB_API_KEY and wandb is not None:
    wandb.login(key=WANDB_API_KEY, relogin=True)
    print({"wandb_login": "ok"})
elif WANDB_API_KEY:
    print({"wandb_login": "skipped_until_requirements_install"})
else:
    print({"wandb_login": "skipped_no_env_key"})

In [ ]:
%cd {REPO_DIR}

bootstrap_command = [
    sys.executable,
    "scripts/init_kaggle.py",
    "--stage",
    "gflownet",
    "--repo-url",
    REPO_URL,
    "--repo-branch",
    REPO_BRANCH,
    "--repo-dir",
    str(REPO_DIR),
    "--dataset-mode",
    DATASET_MODE,
    "--config",
    str(CONFIG_OVERRIDE),
]
if GFLOWNET_CHECKPOINT_DOWNLOAD_SOURCE.strip():
    bootstrap_command.extend([
        "--gflownet-checkpoint-download-source",
        GFLOWNET_CHECKPOINT_DOWNLOAD_SOURCE.strip(),
    ])
run_stream(bootstrap_command, cwd=REPO_DIR)

runtime_report = report_runtime(require_gpu=True)
print(json_dumps({"runtime": runtime_report}))

processed_dataset_ready = all(path.exists() for path in PROCESSED_DATASET_CHECKS.values())
if not processed_dataset_ready:
    download_command = [
        sys.executable,
        "scripts/download_lpm24.py",
        "--output-dir",
        str(LPM24_DATASET_DIR),
    ]
    run_stream(download_command, cwd=REPO_DIR)
else:
    print({"reuse_lpm24_dataset": str(LPM24_DATASET_DIR)})

split_command = [
    sys.executable,
    "scripts/prepare_lpm24_training_splits.py",
    "--input-dir",
    str(LPM24_DATASET_DIR),
    "--validation-fraction",
    str(VALIDATION_FRACTION),
    "--seed",
    str(SPLIT_SEED),
    "--max-target-symbols",
    str(MAX_TARGET_SYMBOLS),
    "--max-stage-symbols",
    str(MAX_STAGE_SYMBOLS),
]
run_stream(split_command, cwd=REPO_DIR)

In [ ]:
import yaml


def apply_ablation_knobs(runtime_config: dict) -> dict:
    tracking_payload = runtime_config.setdefault("tracking", {})
    tracking_payload["run_name"] = RUN_NAME
    existing_tags = list(tracking_payload.get("tags") or [])
    tracking_payload["tags"] = sorted(set(existing_tags + ["lpm24", "gflownet", "ablation", GFLOWNET_VERSION, "parallel_training", ABLATION_NAME]))
    runtime_config.setdefault("training", {})["output_dir"] = str(OUTPUT_DIR)

    gflownet_payload = runtime_config.setdefault("gflownet", {})
    gflownet_payload["objective"] = GFLOWNET_OBJECTIVE
    gflownet_payload["gflownet_iterations"] = int(GFLOWNET_ITERATIONS)
    gflownet_payload["batch_size"] = int(GFLOWNET_BATCH_SIZE)
    gflownet_payload["max_optimization_trajectories_per_iter"] = int(MAX_OPTIMIZATION_TRAJECTORIES_PER_ITER)
    gflownet_payload["scoring_microbatch_size"] = int(GFLOWNET_SCORING_MICROBATCH_SIZE)
    gflownet_payload["learning_rate"] = float(GFLOWNET_LEARNING_RATE)
    gflownet_payload["warmup_ratio"] = float(GFLOWNET_WARMUP_RATIO)
    gflownet_payload["save_every_iterations"] = int(GFLOWNET_SAVE_EVERY_ITERATIONS)
    gflownet_payload["invalid_terminal_reward"] = float(GFLOWNET_INVALID_TERMINAL_REWARD)
    gflownet_payload["parallel_training"] = {
        "enabled": bool(PARALLEL_TRAINING_ENABLED),
        "mode": PARALLEL_TRAINING_MODE,
        "devices": list(PARALLEL_TRAINING_DEVICES),
        "strict": bool(PARALLEL_TRAINING_STRICT),
    }

    target_guidance_payload = dict(gflownet_payload.get("target_guidance", {}))
    target_guidance_payload["enabled"] = bool(TARGET_GUIDANCE_ENABLED)
    target_guidance_payload["on_policy_fraction"] = float(TARGET_GUIDANCE_ON_POLICY_FRACTION)
    target_guidance_payload["target_prefix_rollout_fraction"] = float(TARGET_GUIDANCE_PREFIX_FRACTION)
    target_guidance_payload["target_teacher_fraction"] = float(TARGET_GUIDANCE_TEACHER_FRACTION)
    target_guidance_payload["shuffle_target_selfies_list"] = bool(TARGET_GUIDANCE_SHUFFLE_TARGET_SELFIES_LIST)
    gflownet_payload["target_guidance"] = target_guidance_payload

    replay_payload = dict(gflownet_payload.get("replay", {}))
    replay_payload["enabled"] = bool(REPLAY_ENABLED)
    replay_payload["buffer_type"] = REPLAY_BUFFER_TYPE
    replay_payload["replay_fraction"] = float(REPLAY_FRACTION)
    replay_payload["with_replacement"] = bool(REPLAY_WITH_REPLACEMENT)
    replay_payload["capacity"] = int(REPLAY_CAPACITY)
    replay_payload["recent_fraction"] = float(REPLAY_RECENT_FRACTION)
    replay_payload["reward_fraction"] = float(REPLAY_REWARD_FRACTION)
    replay_payload["uniform_fraction"] = float(REPLAY_UNIFORM_FRACTION)
    replay_payload["tb_residual_fraction"] = float(REPLAY_TB_RESIDUAL_FRACTION)
    replay_payload["reward_temperature"] = float(REPLAY_REWARD_TEMPERATURE)
    replay_payload["tb_residual_temperature"] = float(REPLAY_TB_RESIDUAL_TEMPERATURE)
    replay_payload["recent_window_size"] = int(REPLAY_RECENT_WINDOW_SIZE)
    replay_payload["max_invalid_fraction"] = float(REPLAY_MAX_INVALID_FRACTION)
    replay_payload["max_duplicate_fraction"] = float(REPLAY_MAX_DUPLICATE_FRACTION)
    gflownet_payload["replay"] = replay_payload

    rollout_payload = dict(gflownet_payload.get("rollout", {}))
    rollout_payload["max_stage_new_tokens"] = int(ROLLOUT_MAX_STAGE_NEW_TOKENS)
    rollout_payload["max_molecules_per_sequence"] = int(ROLLOUT_MAX_MOLECULES_PER_SEQUENCE)
    rollout_payload["max_sequence_length"] = int(ROLLOUT_MAX_SEQUENCE_LENGTH)
    rollout_payload["decoding_strategy"] = ROLLOUT_DECODING_STRATEGY
    rollout_payload["num_beams"] = int(ROLLOUT_NUM_BEAMS)
    rollout_payload["length_penalty"] = float(ROLLOUT_LENGTH_PENALTY)
    rollout_payload["early_stopping"] = bool(ROLLOUT_EARLY_STOPPING)
    if FORCE_VALID_MASKING:
        rollout_payload["constrained_decoding"] = True
        rollout_payload["selfies_dict_path"] = SELFIES_DICT_PATH
    rollout_payload["stage_separator"] = " "
    if ROLLOUT_TEMPERATURE is not None:
        rollout_payload["temperature"] = float(ROLLOUT_TEMPERATURE)
    if ROLLOUT_TOP_P is not None:
        rollout_payload["top_p"] = float(ROLLOUT_TOP_P)
    rollout_payload["append_probability"] = float(ROLLOUT_APPEND_PROBABILITY)
    rollout_payload["invalid_append_probability"] = float(ROLLOUT_INVALID_APPEND_PROBABILITY)
    rollout_payload["terminate_on_invalid_stage"] = bool(TERMINATE_ON_INVALID_STAGE)
    gflownet_payload["rollout"] = rollout_payload

    reward_payload = dict(runtime_config.get("reward", {}))
    reward_payload["diversity_beta"] = float(REWARD_DIVERSITY_BETA)
    reward_payload["diversity_weight"] = float(REWARD_DIVERSITY_WEIGHT)
    reward_payload["match_weight"] = float(REWARD_MATCH_WEIGHT)
    reward_payload["reward_variant"] = REWARD_VARIANT
    reward_payload["plus_valid"] = float(REWARD_PLUS_VALID)
    reward_payload["penalty_invalid"] = float(REWARD_PENALTY_INVALID)
    reward_payload["enable_invalid_similarity_ngram_fallback"] = bool(ENABLE_INVALID_SIMILARITY_NGRAM_FALLBACK)
    runtime_config["reward"] = reward_payload
    return runtime_config


TEMP_CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
runtime_config = yaml.safe_load(BASE_TRAIN_CONFIG.read_text())
runtime_config = apply_ablation_knobs(runtime_config)
TEMP_CONFIG_PATH.write_text(yaml.safe_dump(runtime_config, sort_keys=False), encoding="utf-8")

control_summary = {
    "gflownet_version": GFLOWNET_VERSION,
    "repo_branch": REPO_BRANCH,
    "runtime_config": str(TEMP_CONFIG_PATH),
    "output_dir": str(OUTPUT_DIR),
    "gflownet_objective": runtime_config["gflownet"].get("objective"),
    "gflownet_iterations": runtime_config["gflownet"].get("gflownet_iterations"),
    "gflownet_batch_size": runtime_config["gflownet"].get("batch_size"),
    "max_optimization_trajectories_per_iter": runtime_config["gflownet"].get("max_optimization_trajectories_per_iter"),
    "gflownet_scoring_microbatch_size": runtime_config["gflownet"].get("scoring_microbatch_size"),
    "gflownet_learning_rate": runtime_config["gflownet"].get("learning_rate"),
    "gflownet_warmup_ratio": runtime_config["gflownet"].get("warmup_ratio"),
    "rollout_decoding_strategy": runtime_config["gflownet"]["rollout"].get("decoding_strategy"),
    "rollout_num_beams": runtime_config["gflownet"]["rollout"].get("num_beams"),
    "rollout_length_penalty": runtime_config["gflownet"]["rollout"].get("length_penalty"),
    "rollout_early_stopping": runtime_config["gflownet"]["rollout"].get("early_stopping"),
    "target_guidance_enabled": runtime_config["gflownet"]["target_guidance"].get("enabled"),
    "target_guidance_on_policy_fraction": runtime_config["gflownet"]["target_guidance"].get("on_policy_fraction"),
    "target_guidance_prefix_fraction": runtime_config["gflownet"]["target_guidance"].get("target_prefix_rollout_fraction"),
    "target_guidance_teacher_fraction": runtime_config["gflownet"]["target_guidance"].get("target_teacher_fraction"),
    "replay_enabled": runtime_config["gflownet"]["replay"].get("enabled"),
    "replay_buffer_type": runtime_config["gflownet"]["replay"].get("buffer_type"),
    "replay_fraction": runtime_config["gflownet"]["replay"].get("replay_fraction"),
    "replay_recent_fraction": runtime_config["gflownet"]["replay"].get("recent_fraction"),
    "replay_reward_fraction": runtime_config["gflownet"]["replay"].get("reward_fraction"),
    "replay_uniform_fraction": runtime_config["gflownet"]["replay"].get("uniform_fraction"),
    "replay_tb_residual_fraction": runtime_config["gflownet"]["replay"].get("tb_residual_fraction"),
    "reward_variant": runtime_config["reward"].get("reward_variant"),
    "enable_invalid_similarity_ngram_fallback": runtime_config["reward"].get("enable_invalid_similarity_ngram_fallback"),
    "parallel_training_enabled": runtime_config["gflownet"]["parallel_training"].get("enabled"),
    "parallel_training_mode": runtime_config["gflownet"]["parallel_training"].get("mode"),
    "parallel_training_devices": runtime_config["gflownet"]["parallel_training"].get("devices"),
    "parallel_training_strict": runtime_config["gflownet"]["parallel_training"].get("strict"),
}
print(json_dumps(control_summary))

training_command = [
    sys.executable,
    "scripts/train_multi_molecule_gflownet.py",
    "--config",
    str(TEMP_CONFIG_PATH),
    "--output-dir",
    str(OUTPUT_DIR),
]
run_stream(training_command, cwd=REPO_DIR)

In [ ]:
import json
import random

import torch
import wandb
from transformers import AutoTokenizer

from evaluation_metrics import (
    EvaluationMetricConfig,
    GenerationGroup,
    MoleculeInput,
    evaluate_generation_groups,
)
from post_training.gflownet import (
    GFlowNetModel,
    build_gflownet_config,
    build_reward_config,
    sample_stage_trajectories_for_example,
)
from post_training.shared.dataset import MultiMoleculeDataset
from post_training.shared.decoding import build_stage_token_constraints
from src.device import choose_device

EVAL_NUM_EXAMPLES = 128
EVAL_SEED = 42

validation_dataset = MultiMoleculeDataset.from_jsonl(GROUPED_SPLITS_DIR / "validation_multimol.jsonl")
if len(validation_dataset) == 0:
    raise RuntimeError("Validation split is empty; cannot run GFlowNet generation evaluation.")

rng = random.Random(EVAL_SEED)
eval_examples = [
    validation_dataset[rng.randrange(len(validation_dataset))]
    for _ in range(min(EVAL_NUM_EXAMPLES, len(validation_dataset)))
]

summary_payload = json.loads(RUN_SUMMARY_PATH.read_text()) if RUN_SUMMARY_PATH.exists() else {}
best_checkpoint_dir = Path(summary_payload["best_checkpoint_dir"]) if summary_payload.get("best_checkpoint_dir") else BEST_CHECKPOINT_DIR
checkpoint_for_eval = best_checkpoint_dir if best_checkpoint_dir.exists() else BEST_CHECKPOINT_DIR

if not checkpoint_for_eval.exists():
    raise FileNotFoundError(f"Missing checkpoint for evaluation: {checkpoint_for_eval}")

tokenizer = AutoTokenizer.from_pretrained(checkpoint_for_eval, use_fast=True)
tokenizer.model_max_length = int(1e9)

eval_config = yaml.safe_load(TEMP_CONFIG_PATH.read_text())
gflownet_config = build_gflownet_config(eval_config)
reward_config = build_reward_config(
    eval_config.get("reward", {}),
    dataset_hint=str(eval_config.get("data", {}).get("validation_file", "")),
)

device = choose_device(eval_config["training"].get("device", "auto"))
model = GFlowNetModel.from_pretrained(
    checkpoint_for_eval,
    use_lora=gflownet_config.use_lora,
    lora_rank=gflownet_config.lora_rank,
    lora_alpha=gflownet_config.lora_alpha,
    lora_dropout=gflownet_config.lora_dropout,
    target_modules=gflownet_config.target_modules,
    freeze_base_model_without_lora=gflownet_config.freeze_base_model_without_lora,
)
model.to(device)
model.eval()

if gflownet_config.rollout.constrained_decoding:
    constraints = build_stage_token_constraints(
        tokenizer,
        validation_dataset,
        selfies_dict_path=gflownet_config.rollout.selfies_dict_path,
        separator_token=gflownet_config.rollout.stage_separator,
    )
    model.set_stage_token_constraints(constraints)

groups = []
with torch.no_grad():
    for index, example in enumerate(eval_examples):
        trajectories = sample_stage_trajectories_for_example(
            model,
            tokenizer,
            example,
            rollout_id=f"validation-eval-{index:06d}-{example['id']}",
            generation_config=gflownet_config.rollout,
            reward_config=reward_config,
            invalid_terminal_reward=gflownet_config.invalid_terminal_reward,
            device=device,
            rng=rng,
            return_last_valid_trajectory_only=False,
        )

        generated_selfies = [
            trajectory.sampled_selfies
            for trajectory in trajectories
            if trajectory.sampled_selfies
        ]

        groups.append(
            GenerationGroup(
                group_id=str(example["id"]),
                candidates=tuple(
                    MoleculeInput(text=selfies, representation="selfies")
                    for selfies in generated_selfies
                ),
                targets=tuple(
                    MoleculeInput(text=selfies, representation="selfies")
                    for selfies in example["target_selfies_list"]
                ),
            )
        )

result = evaluate_generation_groups(
    groups,
    config=EvaluationMetricConfig(
        acceptance_dice_threshold=0.7,
        compute_n_circles=False,
        n_circles_tanimoto_threshold=0.6,
    ),
)

payload = result.to_dict(include_assessments=False)
EVAL_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
EVAL_OUTPUT_PATH.write_text(json.dumps(payload, indent=2), encoding="utf-8")

evaluation_diagnosis = {
    "eval/accepted_unique_count": result.accepted_unique_count,
    "eval/n_circles": result.n_circles,
    "eval/internal_diversity": result.internal_diversity,
    "eval/novelty_fraction": result.novelty_fraction,
    "eval/novelty_count": result.novelty_count,
    "eval/valid_fraction": result.num_valid_candidates / max(result.num_candidates, 1),
    "eval/mean_max_dice_similarity": result.mean_max_dice_similarity,
}

tracking_config = eval_config.get("tracking", {})
if wandb.run is None and WANDB_API_KEY and tracking_config.get("enabled", True):
    wandb.init(
        project=tracking_config.get("project", "Thesis-2"),
        entity=tracking_config.get("workspace") or None,
        name=f"{RUN_NAME}-validation-eval",
        job_type="validation_evaluation",
    )
if wandb.run is not None:
    wandb.log(evaluation_diagnosis)
else:
    print({"wandb_log": "skipped_no_active_run"})

print(json_dumps(evaluation_diagnosis))
print(json_dumps({"validation_evaluation_metrics_path": str(EVAL_OUTPUT_PATH)}))

In [ ]:
summary_payload = read_json(RUN_SUMMARY_PATH) if RUN_SUMMARY_PATH.exists() else {}
required_outputs = ensure_paths_exist({
    "output_dir": OUTPUT_DIR,
    "best_checkpoint": summary_payload.get("best_checkpoint_dir") or BEST_CHECKPOINT_DIR,
    "run_summary": RUN_SUMMARY_PATH,
    "validation_evaluation_metrics": EVAL_OUTPUT_PATH,
    "runtime_config": TEMP_CONFIG_PATH,
})

stage_root, manifest = export_stage_artifacts(
    stage_name=STAGE_NAME,
    artifact_map={
        "output": OUTPUT_DIR,
        "run_summary.json": RUN_SUMMARY_PATH,
        "validation_evaluation_metrics.json": EVAL_OUTPUT_PATH,
        "runtime_config.yaml": TEMP_CONFIG_PATH,
    },
    metadata={
        "gflownet_version": GFLOWNET_VERSION,
        "repo_branch": REPO_BRANCH,
        "run_name": RUN_NAME,
        "ablation_name": ABLATION_NAME,
        "gflownet_objective": GFLOWNET_OBJECTIVE,
        "parallel_training": {
            "enabled": bool(PARALLEL_TRAINING_ENABLED),
            "mode": PARALLEL_TRAINING_MODE,
            "devices": list(PARALLEL_TRAINING_DEVICES),
            "strict": bool(PARALLEL_TRAINING_STRICT),
        },
        "checkpoint_source_configured": bool(GFLOWNET_CHECKPOINT_DOWNLOAD_SOURCE.strip()),
        "required_outputs": required_outputs,
        "summary": summary_payload,
    },
)
artifact_zip_path = create_zip_archive(stage_root, STAGE_ARTIFACT_ZIP)

print(json_dumps({
    "gflownet_version": GFLOWNET_VERSION,
    "stage_name": STAGE_NAME,
    "stage_root": str(stage_root),
    "stage_root_exists": stage_root.exists(),
    "artifact_zip_path": str(artifact_zip_path),
    "artifact_zip_exists": artifact_zip_path.exists(),
    "output_dir": str(OUTPUT_DIR),
    "output_dir_exists": OUTPUT_DIR.exists(),
    "run_summary": str(RUN_SUMMARY_PATH),
    "run_summary_exists": RUN_SUMMARY_PATH.exists(),
    "validation_evaluation_metrics": str(EVAL_OUTPUT_PATH),
    "validation_evaluation_metrics_exists": EVAL_OUTPUT_PATH.exists(),
    "upstream_checkpoint": str(UPSTREAM_CHECKPOINT),
    "upstream_checkpoint_exists": UPSTREAM_CHECKPOINT.exists(),
    "processed_dataset_checks": {name: path.exists() for name, path in PROCESSED_DATASET_CHECKS.items()},
    "grouped_split_checks": {name: path.exists() for name, path in GROUPED_SPLIT_CHECKS.items()},
    "manifest": manifest,
}))